# Feature Engineering

**OJO: Al finalizar este notebook van a ver, varias clases a predecir llamadas "clase_...". Cuando se seleccione para entrenar se debe seleccionar una y descartar las otras. No incluirlas en las features de entrenamiento!!**

In [1]:
# Importar dataset
import polars as pl
df = pl.read_csv("sell-in.txt.gz", separator="\t")
print(df)

shape: (2_945_818, 7)
┌─────────┬─────────────┬────────────┬─────────────────┬────────────────┬────────────────┬─────────┐
│ periodo ┆ customer_id ┆ product_id ┆ plan_precios_cu ┆ cust_request_q ┆ cust_request_t ┆ tn      │
│ ---     ┆ ---         ┆ ---        ┆ idados          ┆ ty             ┆ n              ┆ ---     │
│ i64     ┆ i64         ┆ i64        ┆ ---             ┆ ---            ┆ ---            ┆ f64     │
│         ┆             ┆            ┆ i64             ┆ i64            ┆ f64            ┆         │
╞═════════╪═════════════╪════════════╪═════════════════╪════════════════╪════════════════╪═════════╡
│ 201701  ┆ 10234       ┆ 20524      ┆ 0               ┆ 2              ┆ 0.053          ┆ 0.053   │
│ 201701  ┆ 10032       ┆ 20524      ┆ 0               ┆ 1              ┆ 0.13628        ┆ 0.13628 │
│ 201701  ┆ 10217       ┆ 20524      ┆ 0               ┆ 1              ┆ 0.03028        ┆ 0.03028 │
│ 201701  ┆ 10125       ┆ 20524      ┆ 0               ┆ 1           

## Definición de granularidad (Producto/Producto-Cliente)

In [2]:
#granularidad:
 # p: producto
 # pc: producto-cliente
granularidad = "pc"   


if granularidad == "p":
    # Por cada producto, se evalua cual es el período más antiguo en el que se vendió este producto para algún cliente. Luego se agrega este dato en una columna llamado "first_sell_in_period" manteniendo el dataset original. Para esto se puede usar la función groupby y agg de polars.
    df = df.with_columns(
        pl.col("periodo").min().over("product_id").alias("first_sell_in_period")
    )  
    print(df)
elif granularidad == "pc":
    df = df.with_columns(
    pl.col("periodo").min().over(["product_id", "customer_id"]).alias("first_sell_in_period")
    )  
    print(df)
else:
    print("Granularidad no válida. Por favor, elija 'p' o 'pc'.")



shape: (2_945_818, 8)
┌─────────┬─────────────┬────────────┬────────────┬────────────┬────────────┬─────────┬────────────┐
│ periodo ┆ customer_id ┆ product_id ┆ plan_preci ┆ cust_reque ┆ cust_reque ┆ tn      ┆ first_sell │
│ ---     ┆ ---         ┆ ---        ┆ os_cuidado ┆ st_qty     ┆ st_tn      ┆ ---     ┆ _in_period │
│ i64     ┆ i64         ┆ i64        ┆ s          ┆ ---        ┆ ---        ┆ f64     ┆ ---        │
│         ┆             ┆            ┆ ---        ┆ i64        ┆ f64        ┆         ┆ i64        │
│         ┆             ┆            ┆ i64        ┆            ┆            ┆         ┆            │
╞═════════╪═════════════╪════════════╪════════════╪════════════╪════════════╪═════════╪════════════╡
│ 201701  ┆ 10234       ┆ 20524      ┆ 0          ┆ 2          ┆ 0.053      ┆ 0.053   ┆ 201701     │
│ 201701  ┆ 10032       ┆ 20524      ┆ 0          ┆ 1          ┆ 0.13628    ┆ 0.13628 ┆ 201701     │
│ 201701  ┆ 10217       ┆ 20524      ┆ 0          ┆ 1          ┆ 0.03

## Completado de 0s y Nulls

In [3]:
# Para cada combinación de producto y cliente se completan los períodos faltantes entre el período más antiguo (first_sell_in_period) y el período más reciente (201912).
# Hay que completar la columna periodo con el periodo faltante correspondiente en el formato AAAAMM.
# Las columnas customer_id, product_id se mantienen constantes para cada combinación de producto y cliente.
# La columna first_sell_in_period se completa con el valor correspondiente a cada par producto-cliente.
# La columna plan_precios_cuidados se toma tal cual viene del dataset cargado, por (producto, mes).
# Las columnas cust_request_qty, cust_request_tn, tn se completan con el valor 0.

# Definimos el límite superior de los periodos como entero
FECHA_LIMITE = 201912

# 1. Creamos la grilla con todos los periodos teóricos en formato AAAAMM
grilla_base = (
    df.select(["customer_id", "product_id", "first_sell_in_period"])
    .unique()
    .filter(pl.col("first_sell_in_period") <= FECHA_LIMITE)
    .with_columns(
        # Convertimos AAAAMM a una escala lineal de meses: (Año * 12) + Mes
        inicio_meses=(pl.col("first_sell_in_period") // 100) * 12 + (pl.col("first_sell_in_period") % 100),
        fin_meses=(FECHA_LIMITE // 100) * 12 + (FECHA_LIMITE % 100)
    )
    .with_columns(
        # Generamos el rango de pasos de meses a rellenar
        pasos=pl.int_ranges(0, pl.col("fin_meses") - pl.col("inicio_meses") + 1)
    )
    .explode("pasos")
)

# Reconstruimos la columna 'periodo' en formato entero AAAAMM
grilla = grilla_base.with_columns(
    mes_actual_lineal=pl.col("inicio_meses") + pl.col("pasos")
).with_columns(
    periodo=(
        ((pl.col("mes_actual_lineal") - 1) // 12) * 100 +
        ((pl.col("mes_actual_lineal") - 1) % 12 + 1)
    ).cast(pl.Int64)
).select(["customer_id", "product_id", "periodo"])

# 1.b plan_precios_cuidados es un atributo de (producto, mes), no del par: dentro de
# un mismo (product_id, periodo) todos los clientes traen el mismo valor. Lo tomamos
# tal cual viene del dataset cargado para poder pegarlo tambien en los meses en que
# este cliente no compro (donde el join por par no trae nada).
ppc_producto_mes = (
    df
    .group_by(["product_id", "periodo"])
    .agg(pl.col("plan_precios_cuidados").first().alias("plan_precios_cuidados"))
)

# 2. Hacemos el Left Join con tus datos originales.
#    plan_precios_cuidados sale del join por (producto, mes), no del join por par.
df_completado = (
    grilla
    .join(df.drop("plan_precios_cuidados"), on=["customer_id", "product_id", "periodo"], how="left")
    .join(ppc_producto_mes, on=["product_id", "periodo"], how="left")
)

# 3. Rellenamos las columnas según tus especificaciones
df_final = df_completado.with_columns(
    # Columnas que van obligatoriamente en 0 si no existía registro
    pl.col("cust_request_qty").fill_null(0),
    pl.col("cust_request_tn").fill_null(0.0),
    pl.col("tn").fill_null(0.0),

    # Constante por par producto-cliente
    pl.col("first_sell_in_period").forward_fill().backward_fill().over(["customer_id", "product_id"])
)
# plan_precios_cuidados NO se rellena: si queda null es porque ningun cliente compro
# ese producto ese mes, o sea que el flag no existe en los datos (~9% de las filas).

# 4. Si tienes más columnas descriptivas (ej: 'match'), las arrastramos dinámicamente
columnas_restantes = [col for col in df.columns if col not in ["customer_id", "product_id", "periodo", "cust_request_qty", "cust_request_tn", "tn", "plan_precios_cuidados", "first_sell_in_period"]]

if columnas_restantes:
    df_final = df_final.with_columns(
        pl.col(columnas_restantes).forward_fill().backward_fill().over(["customer_id", "product_id"])
    )

# Ordenamos el dataframe para su presentación final
df_final = df_final.sort(["product_id", "customer_id", "periodo"])

print(df_final)


shape: (10_096_720, 8)
┌────────────┬────────────┬─────────┬────────────┬────────────┬───────────┬────────────┬───────────┐
│ customer_i ┆ product_id ┆ periodo ┆ cust_reque ┆ cust_reque ┆ tn        ┆ first_sell ┆ plan_prec │
│ d          ┆ ---        ┆ ---     ┆ st_qty     ┆ st_tn      ┆ ---       ┆ _in_period ┆ ios_cuida │
│ ---        ┆ i64        ┆ i64     ┆ ---        ┆ ---        ┆ f64       ┆ ---        ┆ dos       │
│ i64        ┆            ┆         ┆ i64        ┆ f64        ┆           ┆ i64        ┆ ---       │
│            ┆            ┆         ┆            ┆            ┆           ┆            ┆ i64       │
╞════════════╪════════════╪═════════╪════════════╪════════════╪═══════════╪════════════╪═══════════╡
│ 10001      ┆ 20001      ┆ 201701  ┆ 11         ┆ 99.43861   ┆ 99.43861  ┆ 201701     ┆ 0         │
│ 10001      ┆ 20001      ┆ 201702  ┆ 23         ┆ 198.84365  ┆ 198.84365 ┆ 201701     ┆ 0         │
│ 10001      ┆ 20001      ┆ 201703  ┆ 33         ┆ 92.46537   ┆ 92.4

## Aplanado de dataset

In [4]:
# Nos quedamos con los primeros 10 productos y los primeros 10 clientes para testear el pipeline
df_final = df_final.filter(
    (pl.col("product_id") <= 20010) & (pl.col("customer_id") <= 10010)
)

In [5]:
def achatar_dataset(df_final, granularidad=None, t0=201910, max_lags=None, keys=None):
    """
    Achata el dataset (long -> wide): genera lags de tn + ppc0 + la clase.

    Params
    ------
    df_final    : pl.DataFrame ya completado (0s / nulls) con columnas
                  periodo, tn y las columnas de `keys`.
    granularidad: "p" (producto) o "pc" (producto-cliente). Atajo de `keys`.
    t0          : periodo base en formato AAAAMM (default 201910). La clase es t0 + 2.
    max_lags    : cantidad de lags a generar -> columnas tn0..tn{max_lags}.
                  None = todo el historico disponible.
    keys        : lista de columnas de agrupacion. Si viene, gana sobre `granularidad`
                  y permite achatar a cualquier nivel (ej. keys=["cat1"]).

    Returns
    -------
    pl.DataFrame con keys + periodo (= t0) + tn0..tnN + ppc0 + clase_tn.
    Meses previos al first_sell del par quedan en NULL; meses sin venta dentro
    de su vida quedan en 0 (heredado del completado).
    ppc0 = plan_precios_cuidados en t0 (solo el mes base: es lo que se conoce al
    momento de predecir). Se agrega solo si la columna viene en la entrada.
    """
    if keys is None:
        if granularidad == "pc":
            keys = ["product_id", "customer_id"]
        elif granularidad == "p":
            keys = ["product_id"]
        else:
            raise ValueError(
                f"granularidad no soportada: {granularidad!r}. Usa 'p' / 'pc', o pasa keys=[...]"
            )

    # lag = meses hacia atras respecto de t0 (escala lineal para cruzar anios)
    lin_t0 = (t0 // 100) * 12 + (t0 % 100)
    df_lag = df_final.with_columns(
        (lin_t0 - ((pl.col("periodo") // 100) * 12 + (pl.col("periodo") % 100))).alias("lag")
    )

    # Features: sumamos tn por (keys, lag).
    #   - granularidad "p": agrega tn sobre los clientes de cada producto.
    #   - granularidad "pc": es 1 fila por grupo (la suma no cambia nada).
    # Las combinaciones inexistentes (mes previo al first_sell) NO estan en el
    # long, y el pivot con "first" las deja en NULL (no en 0).
    cond = pl.col("lag") >= 0
    if max_lags is not None:
        cond = cond & (pl.col("lag") <= max_lags)

    feats_long = (
        df_lag
        .filter(cond)
        .group_by(keys + ["lag"])
        .agg(pl.col("tn").sum().alias("tn"))
    )

    features = (
        feats_long
        .with_columns(("tn" + pl.col("lag").cast(pl.Int64).cast(pl.String)).alias("lagname"))
        .pivot(values="tn", index=keys, on="lagname", aggregate_function="first")
    )

    # Garantizamos que existan TODAS las columnas tn0..tn{N}, con NULL donde falten
    # (el pivot solo crea columnas para los lags presentes en los datos).
    max_lag = max_lags if max_lags is not None else int(feats_long.select(pl.col("lag").max()).item() or 0)
    faltantes = [
        pl.lit(None, dtype=pl.Float64).alias(f"tn{k}")
        for k in range(max_lag + 1)
        if f"tn{k}" not in features.columns
    ]
    if faltantes:
        features = features.with_columns(faltantes)

    # Clase: tn dos meses adelante de t0 (lag == -2)
    clase = (
        df_lag
        .filter(pl.col("lag") == -2)
        .group_by(keys)
        .agg(pl.col("tn").sum().alias("clase_tn"))
    )

    df_achatado = (
        features
        .join(clase, on=keys, how="left")
        # Guardamos el periodo de referencia (= t0) para identificar / apilar distintos t0
        .with_columns(pl.lit(t0).cast(pl.Int64).alias("periodo"))
    )

    # plan_precios_cuidados del mes base. Condicional: los niveles producto y categoria
    # no traen esta columna, ahi nos quedamos solo con la original de nivel pc.
    extra = []
    if "plan_precios_cuidados" in df_final.columns:
        ppc0 = (
            df_lag
            .filter(pl.col("lag") == 0)
            .group_by(keys)
            .agg(pl.col("plan_precios_cuidados").max().alias("ppc0"))
        )
        df_achatado = df_achatado.join(ppc0, on=keys, how="left")
        extra = ["ppc0"]

    # Ordenamos columnas: keys, periodo, tn0, tn1, ..., ppc0, clase_tn
    lag_cols = sorted(
        [c for c in df_achatado.columns if c.startswith("tn") and c[2:].isdigit()],
        key=lambda c: int(c[2:]),
    )
    return df_achatado.select(keys + ["periodo"] + lag_cols + extra + ["clase_tn"])


# --- Uso ---
# max_lags define cuantos lags generar (tn0..tn{max_lags}); None = todo el historico
df_achatado = achatar_dataset(df_final, granularidad, t0=201910, max_lags=24)
print(df_achatado)


shape: (96, 30)
┌────────────┬─────────────┬─────────┬───────────┬───┬───────────┬───────────┬──────┬───────────┐
│ product_id ┆ customer_id ┆ periodo ┆ tn0       ┆ … ┆ tn23      ┆ tn24      ┆ ppc0 ┆ clase_tn  │
│ ---        ┆ ---         ┆ ---     ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---  ┆ ---       │
│ i64        ┆ i64         ┆ i64     ┆ f64       ┆   ┆ f64       ┆ f64       ┆ i64  ┆ f64       │
╞════════════╪═════════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪══════╪═══════════╡
│ 20005      ┆ 10004       ┆ 201910  ┆ 2.0617    ┆ … ┆ 0.77314   ┆ 3.65951   ┆ 0    ┆ 1.28856   │
│ 20008      ┆ 10005       ┆ 201910  ┆ 0.78419   ┆ … ┆ 103.51341 ┆ 74.49829  ┆ 0    ┆ 0.0       │
│ 20004      ┆ 10009       ┆ 201910  ┆ 54.81469  ┆ … ┆ 120.68031 ┆ 89.80499  ┆ 0    ┆ 38.30299  │
│ 20002      ┆ 10001       ┆ 201910  ┆ 430.90803 ┆ … ┆ 109.11104 ┆ 23.6329   ┆ 0    ┆ 334.03714 │
│ 20007      ┆ 10008       ┆ 201910  ┆ 57.21077  ┆ … ┆ 11.44217  ┆ 21.24973  ┆ 0    ┆ 15.52863  │
│ … 

In [6]:
def normalizar_achatado(df_achatado, metodo="recta", norm_lags=None,
                        prefix="", clase_col="clase_tn", b0_name="B0", b1_name="B1"):
    """
    Normaliza fila a fila las columnas {prefix}tn* (y opcionalmente {clase_col})
    de df_achatado.

    Los parametros de normalizacion (B0, B1) se calculan SOLO con la ventana
    {prefix}tn0..{prefix}tn{norm_lags} (nunca con la clase -> evita leakage) y
    luego se aplican a todos los {prefix}tn* y a la clase, generando
    {prefix}tn0_norm..{prefix}tnN_norm y {clase_col}_norm.

    Params
    ------
    metodo:
      "recta"   -> ajuste lineal por minimos cuadrados sobre la ventana.
                   B0 = ordenada al origen, B1 = pendiente.
                   norm = valor - (B0 + B1 * lag)   (la clase esta en lag = -2)
      "zscore"  -> B0 = media, B1 = desvio.   norm = (valor - B0) / B1
      "minmax"  -> B0 = min,   B1 = max-min.  norm = (valor - B0) / B1
      "media"   -> B0 = 0,     B1 = media.    norm = valor / B1
    norm_lags:
      hasta que lag usar para calcular B0/B1 (ventana {prefix}tn0..{prefix}tn{norm_lags}).
      None = usa todos los lags disponibles.
    prefix:
      prefijo de las columnas de lags a normalizar (ej. "" para pc, "p_" para producto).
    clase_col:
      nombre de la columna de clase a normalizar; None si no hay clase (ej. features
      de producto, donde no se normaliza ninguna clase).
    b0_name, b1_name:
      nombres con los que se guardan los parametros de normalizacion (ej. "B0"/"B1"
      para pc, "p_B0"/"p_B1" para producto).

    Devuelve df_achatado + columnas b0_name, b1_name, {prefix}tn*_norm y (si aplica)
    {clase_col}_norm.
    """
    p = len(prefix)
    lag_cols = sorted(
        [c for c in df_achatado.columns if c.startswith(prefix + "tn") and c[p + 2:].isdigit()],
        key=lambda c: int(c[p + 2:]),
    )
    n_max = int(lag_cols[-1][p + 2:])
    if norm_lags is None:
        norm_lags = n_max
    fit_cols = [f"{prefix}tn{k}" for k in range(norm_lags + 1)]

    if metodo == "recta":
        # Minimos cuadrados sobre la ventana, ignorando los nulls
        n   = pl.sum_horizontal([pl.col(c).is_not_null().cast(pl.Float64) for c in fit_cols])
        Sy  = pl.sum_horizontal([pl.col(c) for c in fit_cols])
        Sx  = pl.sum_horizontal([pl.when(pl.col(c).is_not_null()).then(float(k)).otherwise(0.0)
                                 for k, c in enumerate(fit_cols)])
        Sxx = pl.sum_horizontal([pl.when(pl.col(c).is_not_null()).then(float(k * k)).otherwise(0.0)
                                 for k, c in enumerate(fit_cols)])
        Sxy = pl.sum_horizontal([pl.when(pl.col(c).is_not_null()).then(pl.col(c) * float(k)).otherwise(0.0)
                                 for k, c in enumerate(fit_cols)])
        denom = n * Sxx - Sx * Sx
        slope = pl.when(denom != 0).then((n * Sxy - Sx * Sy) / denom).otherwise(0.0)
        inter = pl.when(n > 0).then((Sy - slope * Sx) / n).otherwise(None)

        df = df_achatado.with_columns(inter.alias(b0_name), slope.alias(b1_name))

        # norm = valor - (B0 + B1 * lag);  la clase esta en lag = -2 (t0 + 2)
        exprs = [
            (pl.col(c) - (pl.col(b0_name) + pl.col(b1_name) * float(int(c[p + 2:])))).alias(f"{c}_norm")
            for c in lag_cols
        ]
        if clase_col is not None:
            exprs.append((pl.col(clase_col) - (pl.col(b0_name) + pl.col(b1_name) * (-2.0))).alias(f"{clase_col}_norm"))
        return df.with_columns(exprs)

    # --- metodos afines: norm = (valor - B0) / B1 ---
    if metodo == "zscore":
        B0 = pl.mean_horizontal(fit_cols)
        B1 = pl.concat_list(fit_cols).list.std()
    elif metodo == "minmax":
        B0 = pl.min_horizontal(fit_cols)
        B1 = pl.max_horizontal(fit_cols) - pl.min_horizontal(fit_cols)
    elif metodo == "media":
        B0 = pl.lit(0.0)
        B1 = pl.mean_horizontal(fit_cols)
    else:
        raise ValueError(f"metodo no soportado: {metodo}")

    df = df_achatado.with_columns(B0.alias(b0_name), B1.alias(b1_name))
    # B1 seguro para no dividir por 0 ni por null (series constantes / sin datos)
    b1_safe = pl.when((pl.col(b1_name) == 0) | pl.col(b1_name).is_null()).then(1.0).otherwise(pl.col(b1_name))
    exprs = [((pl.col(c) - pl.col(b0_name)) / b1_safe).alias(f"{c}_norm") for c in lag_cols]
    if clase_col is not None:
        exprs.append(((pl.col(clase_col) - pl.col(b0_name)) / b1_safe).alias(f"{clase_col}_norm"))
    return df.with_columns(exprs)


# --- Uso ---
# metodo: "recta" | "zscore" | "minmax" | "media"
# norm_lags: ventana tn0..tn{norm_lags} para calcular B0/B1 (None = todos los lags)
df_norm = normalizar_achatado(df_achatado, metodo="media", norm_lags=None)
print(df_norm)

shape: (96, 58)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ product_id ┆ customer_ ┆ periodo ┆ tn0       ┆ … ┆ tn22_norm ┆ tn23_norm ┆ tn24_norm ┆ clase_tn_ │
│ ---        ┆ id        ┆ ---     ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ norm      │
│ i64        ┆ ---       ┆ i64     ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ ---       │
│            ┆ i64       ┆         ┆           ┆   ┆           ┆           ┆           ┆ f64       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 20005      ┆ 10004     ┆ 201910  ┆ 2.0617    ┆ … ┆ 2.253484  ┆ 0.346395  ┆ 1.639597  ┆ 0.577323  │
│ 20008      ┆ 10005     ┆ 201910  ┆ 0.78419   ┆ … ┆ 0.882815  ┆ 3.236987  ┆ 2.329649  ┆ 0.0       │
│ 20004      ┆ 10009     ┆ 201910  ┆ 54.81469  ┆ … ┆ 2.023003  ┆ 3.365417  ┆ 2.504395  ┆ 1.068157  │
│ 20002      ┆ 10001     ┆ 201910  ┆ 430.90803 ┆ … ┆ 0.941347  ┆ 0.551131  

In [7]:
def desnormalizar(df, col_norm, metodo, lag=-2, alias="pred_tn"):
    """
    Recupera la escala original (toneladas) de una columna normalizada,
    usando los parametros B0/B1 guardados por normalizar_achatado.
    Es el inverso exacto de normalizar_achatado.

    Params
    ------
    df       : DataFrame que tiene las columnas B0, B1 y col_norm.
    col_norm : columna normalizada a desnormalizar (ej. la prediccion del modelo).
    metodo   : el MISMO metodo usado al normalizar ("recta"|"zscore"|"minmax"|"media").
    lag      : posicion temporal de la columna respecto de t0.
               clase_tn (y su prediccion) esta en lag = -2 (t0 + 2); tn{k} esta en lag = k.
    alias    : nombre de la columna de salida.

    Devuelve df + columna 'alias' en escala original.
    """
    if metodo == "recta":
        # inverso de: norm = valor - (B0 + B1 * lag)
        valor = pl.col(col_norm) + (pl.col("B0") + pl.col("B1") * float(lag))
    elif metodo in ("zscore", "minmax", "media"):
        # inverso de: norm = (valor - B0) / B1  (con el mismo B1 seguro del forward)
        b1_safe = pl.when((pl.col("B1") == 0) | pl.col("B1").is_null()).then(1.0).otherwise(pl.col("B1"))
        valor = pl.col(col_norm) * b1_safe + pl.col("B0")
    else:
        raise ValueError(f"metodo no soportado: {metodo}")
    return df.with_columns(valor.alias(alias))


# --- Uso ---
# Chequeo round-trip: desnormalizar clase_tn_norm (lag=-2) tiene que devolver clase_tn.
METODO = "media"
df_check = desnormalizar(df_norm, "clase_tn_norm", METODO, lag=-2, alias="clase_tn_rec")
error_max = df_check.select((pl.col("clase_tn_rec") - pl.col("clase_tn")).abs().max()).item()
print("Error maximo de round-trip:", error_max)

# Cuando tengas la prediccion del modelo en escala normalizada (ej. columna 'clase_tn_norm_pred'),
# la pasas a toneladas asi:
# df_pred = desnormalizar(df_pred, "clase_tn_norm_pred", METODO, lag=-2, alias="tn_pred")


Error maximo de round-trip: 1.4210854715202004e-14


In [8]:
def agregar_deltas(df_norm, salto=2, prefix="", clase_col="clase_tn"):
    """
    Agrega columnas de delta (saltos de 'salto' periodos) sobre las columnas
    NORMALIZADAS. No elimina ninguna columna, solo agrega.

      {prefix}tn{k}_delta = {prefix}tn{k}_norm - {prefix}tn{k+salto}_norm  (k = 0 .. maxlag-salto)
      {clase_col}_delta   = {clase_col}_norm - {prefix}tn0_norm            (cambio a 2 periodos -> target)

    tn0_delta es el delta t0 - t2 (el cambio en el ti mas reciente).
    Para este problema salto=2 porque se predice a 2 periodos.

    Params
    ------
    prefix:
      prefijo de las columnas normalizadas a deltear (ej. "" para pc, "p_" para producto).
    clase_col:
      nombre base de la clase a deltear; None si no hay clase (ej. nivel producto).
    """
    p = len(prefix)
    norm_lags = sorted(
        int(c[p + 2:-5]) for c in df_norm.columns
        if c.startswith(prefix + "tn") and c.endswith("_norm") and c[p + 2:-5].isdigit()
    )
    max_k = max(norm_lags)
    exprs = [
        (pl.col(f"{prefix}tn{k}_norm") - pl.col(f"{prefix}tn{k + salto}_norm")).alias(f"{prefix}tn{k}_delta")
        for k in range(0, max_k - salto + 1)
    ]
    # Delta de la clase: valor de la clase normalizada menos t0 normalizado
    if clase_col is not None:
        exprs.append((pl.col(f"{clase_col}_norm") - pl.col(f"{prefix}tn0_norm")).alias(f"{clase_col}_delta"))
    return df_norm.with_columns(exprs)


# --- Uso ---
df_norm = agregar_deltas(df_norm, salto=2)
print(df_norm)


shape: (96, 82)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ product_id ┆ customer_ ┆ periodo ┆ tn0       ┆ … ┆ tn20_delt ┆ tn21_delt ┆ tn22_delt ┆ clase_tn_ │
│ ---        ┆ id        ┆ ---     ┆ ---       ┆   ┆ a         ┆ a         ┆ a         ┆ delta     │
│ i64        ┆ ---       ┆ i64     ┆ f64       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ i64       ┆         ┆           ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 20005      ┆ 10004     ┆ 201910  ┆ 2.0617    ┆ … ┆ -1.264339 ┆ -0.232854 ┆ 0.613887  ┆ -0.346395 │
│ 20008      ┆ 10005     ┆ 201910  ┆ 0.78419   ┆ … ┆ -0.171658 ┆ -3.187941 ┆ -1.446835 ┆ -0.024523 │
│ 20004      ┆ 10009     ┆ 201910  ┆ 54.81469  ┆ … ┆ -1.813702 ┆ -2.990118 ┆ -0.481393 ┆ -0.460462 │
│ 20002      ┆ 10001     ┆ 201910  ┆ 430.90803 ┆ … ┆ -0.236139 ┆ -0.169834 

In [9]:
df_norm.columns

['product_id',
 'customer_id',
 'periodo',
 'tn0',
 'tn1',
 'tn2',
 'tn3',
 'tn4',
 'tn5',
 'tn6',
 'tn7',
 'tn8',
 'tn9',
 'tn10',
 'tn11',
 'tn12',
 'tn13',
 'tn14',
 'tn15',
 'tn16',
 'tn17',
 'tn18',
 'tn19',
 'tn20',
 'tn21',
 'tn22',
 'tn23',
 'tn24',
 'ppc0',
 'clase_tn',
 'B0',
 'B1',
 'tn0_norm',
 'tn1_norm',
 'tn2_norm',
 'tn3_norm',
 'tn4_norm',
 'tn5_norm',
 'tn6_norm',
 'tn7_norm',
 'tn8_norm',
 'tn9_norm',
 'tn10_norm',
 'tn11_norm',
 'tn12_norm',
 'tn13_norm',
 'tn14_norm',
 'tn15_norm',
 'tn16_norm',
 'tn17_norm',
 'tn18_norm',
 'tn19_norm',
 'tn20_norm',
 'tn21_norm',
 'tn22_norm',
 'tn23_norm',
 'tn24_norm',
 'clase_tn_norm',
 'tn0_delta',
 'tn1_delta',
 'tn2_delta',
 'tn3_delta',
 'tn4_delta',
 'tn5_delta',
 'tn6_delta',
 'tn7_delta',
 'tn8_delta',
 'tn9_delta',
 'tn10_delta',
 'tn11_delta',
 'tn12_delta',
 'tn13_delta',
 'tn14_delta',
 'tn15_delta',
 'tn16_delta',
 'tn17_delta',
 'tn18_delta',
 'tn19_delta',
 'tn20_delta',
 'tn21_delta',
 'tn22_delta',
 'clase_tn_de

In [10]:
#checkeo
df_norm['product_id', 'customer_id', 'periodo', 'tn0_norm', 'tn1_norm','tn2_norm','tn3_norm','tn0_delta',
 'tn1_delta','clase_tn_norm','clase_tn_delta']

product_id,customer_id,periodo,tn0_norm,tn1_norm,tn2_norm,tn3_norm,tn0_delta,tn1_delta,clase_tn_norm,clase_tn_delta
i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64
20005,10004,201910,0.923718,2.251557,0.404125,1.587638,0.519593,0.663919,0.577323,-0.346395
20008,10005,201910,0.024523,0.971096,0.049045,0.078473,-0.024523,0.892623,0.0,-0.024523
20004,10009,201910,1.528619,1.392213,0.102485,0.340656,1.426134,1.051557,1.068157,-0.460462
20002,10001,201910,2.176561,1.077711,0.752165,0.520873,1.424396,0.556838,1.687256,-0.489305
20007,10008,201910,2.679993,0.251592,0.650855,0.765714,2.029138,-0.514122,0.727426,-1.952567
…,…,…,…,…,…,…,…,…,…,…
20007,10006,201910,0.567096,0.492788,0.492786,0.67074,0.07431,-0.177952,0.251279,-0.315816
20009,10008,201910,6.37775,0.85475,1.11775,1.4465,5.26,-0.59175,0.0,-6.37775
20002,10004,201910,2.888497,0.240195,0.049271,0.135494,2.839226,0.1047,1.456006,-1.432491


Agrego a la granularidad pc, la de p para complementar el dataset

In [11]:
df_aux = pl.read_csv("sell-in.txt.gz", separator="\t")

In [12]:
df_aux = df_aux.with_columns(
        pl.col("periodo").min().over("product_id").alias("first_sell_in_period")
    )  
print(df_aux)

shape: (2_945_818, 8)
┌─────────┬─────────────┬────────────┬────────────┬────────────┬────────────┬─────────┬────────────┐
│ periodo ┆ customer_id ┆ product_id ┆ plan_preci ┆ cust_reque ┆ cust_reque ┆ tn      ┆ first_sell │
│ ---     ┆ ---         ┆ ---        ┆ os_cuidado ┆ st_qty     ┆ st_tn      ┆ ---     ┆ _in_period │
│ i64     ┆ i64         ┆ i64        ┆ s          ┆ ---        ┆ ---        ┆ f64     ┆ ---        │
│         ┆             ┆            ┆ ---        ┆ i64        ┆ f64        ┆         ┆ i64        │
│         ┆             ┆            ┆ i64        ┆            ┆            ┆         ┆            │
╞═════════╪═════════════╪════════════╪════════════╪════════════╪════════════╪═════════╪════════════╡
│ 201701  ┆ 10234       ┆ 20524      ┆ 0          ┆ 2          ┆ 0.053      ┆ 0.053   ┆ 201701     │
│ 201701  ┆ 10032       ┆ 20524      ┆ 0          ┆ 1          ┆ 0.13628    ┆ 0.13628 ┆ 201701     │
│ 201701  ┆ 10217       ┆ 20524      ┆ 0          ┆ 1          ┆ 0.03

In [13]:
# Colapsamos la dimension cliente: agregamos por periodo y producto.
#   - cust_request_qty, cust_request_tn, tn -> suma (acumulado sobre los clientes)
#   - first_sell_in_period                 -> minimo del grupo
# plan_precios_cuidados NO se agrega a nivel producto: nos quedamos solo con la
# columna original (nivel producto-cliente), que es la mas realista.
df_aux = (
    df_aux
    .group_by(["periodo", "product_id"])
    .agg(
        pl.col("cust_request_qty").sum().alias("cust_request_qty"),
        pl.col("cust_request_tn").sum().alias("cust_request_tn"),
        pl.col("tn").sum().alias("tn"),
        pl.col("first_sell_in_period").min().alias("first_sell_in_period"),
    )
    .sort(["product_id", "periodo"])
)
print(df_aux)

shape: (31_243, 6)
┌─────────┬────────────┬──────────────────┬─────────────────┬────────────┬──────────────────────┐
│ periodo ┆ product_id ┆ cust_request_qty ┆ cust_request_tn ┆ tn         ┆ first_sell_in_period │
│ ---     ┆ ---        ┆ ---              ┆ ---             ┆ ---        ┆ ---                  │
│ i64     ┆ i64        ┆ i64              ┆ f64             ┆ f64        ┆ i64                  │
╞═════════╪════════════╪══════════════════╪═════════════════╪════════════╪══════════════════════╡
│ 201701  ┆ 20001      ┆ 479              ┆ 937.72717       ┆ 934.77222  ┆ 201701               │
│ 201702  ┆ 20001      ┆ 432              ┆ 833.72187       ┆ 798.0162   ┆ 201701               │
│ 201703  ┆ 20001      ┆ 509              ┆ 1330.74697      ┆ 1303.35771 ┆ 201701               │
│ 201704  ┆ 20001      ┆ 279              ┆ 1132.9443       ┆ 1069.9613  ┆ 201701               │
│ 201705  ┆ 20001      ┆ 701              ┆ 1550.68936      ┆ 1502.20132 ┆ 201701               │
│

In [14]:
# --- Features a nivel PRODUCTO (complemento "p" de la granularidad pc) ---
# Reutilizamos achatar_dataset sobre df_aux (tn total del producto, sumado sobre
# TODOS los clientes) para obtener las mismas ventanas de lags tn0..tn{max_lags},
# y las pegamos a df_norm por product_id + periodo (= t0).
# Ojo: usamos df_aux (dataset completo) y NO df_final, que esta filtrado a 10
# productos/clientes -> ahi el total por producto seria parcial.

T0 = 201910
MAX_LAGS = 24

# Lags de tn a nivel producto; descartamos la clase (solo t0 y meses anteriores).
prod_feats = achatar_dataset(df_aux, granularidad="p", t0=T0, max_lags=MAX_LAGS)

lag_cols_p = [c for c in prod_feats.columns if c.startswith("tn") and c[2:].isdigit()]
prod_feats = (
    prod_feats
    .drop("clase_tn")
    # tn{k} -> p_tn{k} para no pisar los tn{k} de nivel producto-cliente
    .rename({c: f"p_{c}" for c in lag_cols_p})
    # mes sin ventas del producto -> 0 tn (a nivel producto un mes ausente = 0)
    .with_columns([pl.col(f"p_{c}").fill_null(0.0) for c in lag_cols_p])
)

# Pegamos las features de producto a cada fila (producto-cliente) de df_norm
df_norm = df_norm.join(prod_feats, on=["product_id", "periodo"], how="left")

print(df_norm.select(
    ["product_id", "customer_id", "periodo"] + [f"p_tn{k}" for k in range(MAX_LAGS + 1)]
))

shape: (96, 28)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ product_id ┆ customer_ ┆ periodo ┆ p_tn0     ┆ … ┆ p_tn21    ┆ p_tn22    ┆ p_tn23    ┆ p_tn24    │
│ ---        ┆ id        ┆ ---     ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ i64        ┆ ---       ┆ i64     ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
│            ┆ i64       ┆         ┆           ┆   ┆           ┆           ┆           ┆           │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 20005      ┆ 10004     ┆ 201910  ┆ 996.78275 ┆ … ┆ 417.53208 ┆ 329.42894 ┆ 776.76542 ┆ 875.13411 │
│ 20008      ┆ 10005     ┆ 201910  ┆ 452.77197 ┆ … ┆ 469.29224 ┆ 543.14221 ┆ 631.8606  ┆ 723.19292 │
│ 20004      ┆ 10009     ┆ 201910  ┆ 1064.6963 ┆ … ┆ 415.52538 ┆ 568.79679 ┆ 897.42607 ┆ 1268.2120 │
│            ┆           ┆         ┆ 3         ┆   ┆           ┆           

In [15]:
# Normalizamos los lags de PRODUCTO con la misma funcion/metodo que los pc,
# pero referenciados al producto: parametros propios p_B0 / p_B1 y columnas
# p_tn*_norm. Sin clase (clase_col=None): a nivel producto no normalizamos ninguna.
# metodo debe ser el mismo que en la normalizacion pc de arriba ("media").
df_norm = normalizar_achatado(
    df_norm, metodo="media", norm_lags=None,
    prefix="p_", clase_col=None, b0_name="p_B0", b1_name="p_B1",
)

print(df_norm.select(
    ["product_id", "customer_id", "periodo", "p_B0", "p_B1"]
    + [f"p_tn{k}_norm" for k in range(MAX_LAGS + 1)]
))

shape: (96, 30)
┌────────────┬────────────┬─────────┬──────┬───┬────────────┬────────────┬────────────┬────────────┐
│ product_id ┆ customer_i ┆ periodo ┆ p_B0 ┆ … ┆ p_tn21_nor ┆ p_tn22_nor ┆ p_tn23_nor ┆ p_tn24_nor │
│ ---        ┆ d          ┆ ---     ┆ ---  ┆   ┆ m          ┆ m          ┆ m          ┆ m          │
│ i64        ┆ ---        ┆ i64     ┆ f64  ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│            ┆ i64        ┆         ┆      ┆   ┆ f64        ┆ f64        ┆ f64        ┆ f64        │
╞════════════╪════════════╪═════════╪══════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 20005      ┆ 10004      ┆ 201910  ┆ 0.0  ┆ … ┆ 0.663468   ┆ 0.52347    ┆ 1.234298   ┆ 1.390608   │
│ 20008      ┆ 10005      ┆ 201910  ┆ 0.0  ┆ … ┆ 0.904113   ┆ 1.046388   ┆ 1.217309   ┆ 1.393264   │
│ 20004      ┆ 10009      ┆ 201910  ┆ 0.0  ┆ … ┆ 0.620398   ┆ 0.84924    ┆ 1.339898   ┆ 1.893499   │
│ 20002      ┆ 10001      ┆ 201910  ┆ 0.0  ┆ … ┆ 0.909455   ┆ 0.757807   ┆ 

In [16]:
# Deltas de los lags de PRODUCTO: misma idea que los pc pero sobre p_tn*_norm.
#   p_tn{k}_delta = p_tn{k}_norm - p_tn{k+2}_norm
# Sin clase (clase_col=None): la clase vive solo a nivel producto-cliente.
df_norm = agregar_deltas(df_norm, salto=2, prefix="p_", clase_col=None)

print(df_norm.select(
    ["product_id", "customer_id", "periodo"]
    + [f"p_tn{k}_norm" for k in range(MAX_LAGS + 1)]
    + [f"p_tn{k}_delta" for k in range(MAX_LAGS - 1)]
))


shape: (96, 51)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ product_id ┆ customer_ ┆ periodo ┆ p_tn0_nor ┆ … ┆ p_tn19_de ┆ p_tn20_de ┆ p_tn21_de ┆ p_tn22_de │
│ ---        ┆ id        ┆ ---     ┆ m         ┆   ┆ lta       ┆ lta       ┆ lta       ┆ lta       │
│ i64        ┆ ---       ┆ i64     ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ i64       ┆         ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 20005      ┆ 10004     ┆ 201910  ┆ 1.583911  ┆ … ┆ 0.226364  ┆ 0.110882  ┆ -0.57083  ┆ -0.867138 │
│ 20008      ┆ 10005     ┆ 201910  ┆ 0.872286  ┆ … ┆ 0.5716    ┆ -0.071067 ┆ -0.313195 ┆ -0.346876 │
│ 20004      ┆ 10009     ┆ 201910  ┆ 1.58964   ┆ … ┆ 0.109589  ┆ -0.097262 ┆ -0.7195   ┆ -1.044259 │
│ 20002      ┆ 10001     ┆ 201910  ┆ 1.828084  ┆ … ┆ -0.016569 ┆ -0.100281 

In [17]:
len(df_norm.columns)

157

In [18]:
df_norm.columns

['product_id',
 'customer_id',
 'periodo',
 'tn0',
 'tn1',
 'tn2',
 'tn3',
 'tn4',
 'tn5',
 'tn6',
 'tn7',
 'tn8',
 'tn9',
 'tn10',
 'tn11',
 'tn12',
 'tn13',
 'tn14',
 'tn15',
 'tn16',
 'tn17',
 'tn18',
 'tn19',
 'tn20',
 'tn21',
 'tn22',
 'tn23',
 'tn24',
 'ppc0',
 'clase_tn',
 'B0',
 'B1',
 'tn0_norm',
 'tn1_norm',
 'tn2_norm',
 'tn3_norm',
 'tn4_norm',
 'tn5_norm',
 'tn6_norm',
 'tn7_norm',
 'tn8_norm',
 'tn9_norm',
 'tn10_norm',
 'tn11_norm',
 'tn12_norm',
 'tn13_norm',
 'tn14_norm',
 'tn15_norm',
 'tn16_norm',
 'tn17_norm',
 'tn18_norm',
 'tn19_norm',
 'tn20_norm',
 'tn21_norm',
 'tn22_norm',
 'tn23_norm',
 'tn24_norm',
 'clase_tn_norm',
 'tn0_delta',
 'tn1_delta',
 'tn2_delta',
 'tn3_delta',
 'tn4_delta',
 'tn5_delta',
 'tn6_delta',
 'tn7_delta',
 'tn8_delta',
 'tn9_delta',
 'tn10_delta',
 'tn11_delta',
 'tn12_delta',
 'tn13_delta',
 'tn14_delta',
 'tn15_delta',
 'tn16_delta',
 'tn17_delta',
 'tn18_delta',
 'tn19_delta',
 'tn20_delta',
 'tn21_delta',
 'tn22_delta',
 'clase_tn_de

In [19]:
# acá podría tener lo de los productos por categoría antes de seguir

In [20]:
# --- Toneladas de la jerarquia (cat1, cat2, cat3) en t0 ---
# Misma idea que agregar_toneladas_jerarquicas del otro notebook, pero SOLO para t0:
# aca el dataset ya esta achatado (una fila por par en t0), asi que no hay dimension
# mes que recorrer -> alcanza con el total de cada categoria en el periodo base.
#
# Ojo: se calcula sobre df_aux (dataset completo, tn ya sumado sobre los clientes) y
# NO sobre df_final, que esta filtrado a 10 productos: ahi el total de la categoria
# seria parcial.

df_productos = pl.read_csv("tb_productos.txt", separator="\t")
CATS = ["cat1", "cat2", "cat3"]

# Categoria de cada producto. tb_productos tiene 1 fila por product_id -> el join no
# multiplica filas. Los 45 product_id del sell-in que no estan en tb_productos pasan
# a ser su propia categoria (una de uno): les ponemos una etiqueta unica por producto
# en los 3 niveles, asi su tn_total_cat* termina siendo su propio tn.
prod_cat = (
    df_aux
    .select("product_id")
    .unique()
    .join(df_productos.select(["product_id"] + CATS), on="product_id", how="left")
    .with_columns([
        pl.col(c).fill_null(pl.format("prod_{}", pl.col("product_id"))).alias(c)
        for c in CATS
    ])
)

# Ventas de TODO el mercado en t0, con la categoria de cada producto
ventas_t0 = (
    df_aux
    .filter(pl.col("periodo") == T0)
    .join(prod_cat, on="product_id", how="left")
)

# Pegamos la categoria a cada fila y, por nivel, el tn total de esa categoria en t0.
# El total incluye al propio producto (es el tamanio del mercado de esa categoria).
df_norm = df_norm.join(prod_cat, on="product_id", how="left")
for c in CATS:
    tn_cat = ventas_t0.group_by(c).agg(pl.col("tn").sum().alias(f"tn_total_{c}"))
    df_norm = df_norm.join(tn_cat, on=c, how="left")

# Una categoria sin ventas en t0 vendio 0 tn; no es un dato faltante. Mismo criterio
# que los p_tn* de nivel producto (mes ausente = 0 tn). Afecta sobre todo a las
# etiquetas prod_* de los 45 sin categoria, que en su mayoria no vendieron en t0.
df_norm = df_norm.with_columns([pl.col(f"tn_total_{c}").fill_null(0.0) for c in CATS])

print(df_norm.select(
    ["product_id", "customer_id", "periodo"] + CATS + [f"tn_total_{c}" for c in CATS]
))


shape: (96, 9)
┌────────────┬────────────┬─────────┬───────┬───┬────────────┬────────────┬────────────┬───────────┐
│ product_id ┆ customer_i ┆ periodo ┆ cat1  ┆ … ┆ cat3       ┆ tn_total_c ┆ tn_total_c ┆ tn_total_ │
│ ---        ┆ d          ┆ ---     ┆ ---   ┆   ┆ ---        ┆ at1        ┆ at2        ┆ cat3      │
│ i64        ┆ ---        ┆ i64     ┆ str   ┆   ┆ str        ┆ ---        ┆ ---        ┆ ---       │
│            ┆ i64        ┆         ┆       ┆   ┆            ┆ f64        ┆ f64        ┆ f64       │
╞════════════╪════════════╪═════════╪═══════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ 20005      ┆ 10004      ┆ 201910  ┆ FOODS ┆ … ┆ Mayonesa   ┆ 8702.57576 ┆ 6739.40854 ┆ 5040.9998 │
│ 20008      ┆ 10005      ┆ 201910  ┆ HC    ┆ … ┆ Opaco      ┆ 20473.8573 ┆ 3395.16148 ┆ 1018.1434 │
│            ┆            ┆         ┆       ┆   ┆            ┆ 6          ┆            ┆ 2         │
│ 20004      ┆ 10009      ┆ 201910  ┆ FOODS ┆ … ┆ Mayonesa   ┆ 8702.57576 ┆ 

In [21]:
# --- Achatado por CATEGORIA (cat1, cat2, cat3) ---
# Mismo pipeline que el nivel producto, pero con keys=[cat]: reusamos las tres
# funciones tal cual (achatar -> normalizar -> deltas), solo cambia el prefijo.
#
# Los tres niveles aportan: cat3 agrupa 5 productos en mediana (max 134), asi que su
# serie no es la del producto suelto. Solo 12 de los 1233 productos (1%) son unicos en
# su cat3; para esos c3_tn* si termina siendo igual a p_tn*.

NIVELES = [("cat1", "c1_"), ("cat2", "c2_"), ("cat3", "c3_")]

for cat, pref in NIVELES:
    # Serie mensual de la categoria: tn de todo el mercado sumado sobre sus productos.
    # Sale de df_aux (dataset completo), no de df_final (filtrado a 10 productos).
    df_cat = (
        df_aux
        .join(prod_cat, on="product_id", how="left")
        .group_by([cat, "periodo"])
        .agg(pl.col("tn").sum().alias("tn"))
    )

    # Lags de la categoria. Sin clase: la clase vive solo a nivel producto-cliente.
    cat_feats = achatar_dataset(df_cat, keys=[cat], t0=T0, max_lags=MAX_LAGS).drop("clase_tn")
    lag_cols_c = [c for c in cat_feats.columns if c.startswith("tn") and c[2:].isdigit()]
    # tn{k} -> {pref}tn{k} para no pisar los tn{k} de nivel producto-cliente
    cat_feats = cat_feats.rename({c: f"{pref}{c}" for c in lag_cols_c})

    df_norm = df_norm.join(cat_feats, on=[cat, "periodo"], how="left")

    # Dos fuentes de null, y las dos significan 0 tn (mismo criterio que p_tn*):
    #   - mes sin ventas de la categoria -> el pivot lo deja null
    #   - categoria sin ventas en TODA la ventana t0..t0-max_lags -> no entra al
    #     achatado y el left join no la encuentra (pasa con las etiquetas prod_*)
    # Rellenamos despues del join para cubrir las dos, y antes de normalizar para
    # que B0/B1 no salgan nulos.
    df_norm = df_norm.with_columns(
        [pl.col(f"{pref}tn{k}").fill_null(0.0) for k in range(MAX_LAGS + 1)]
    )

    df_norm = normalizar_achatado(
        df_norm, metodo="media", norm_lags=None,
        prefix=pref, clase_col=None, b0_name=f"{pref}B0", b1_name=f"{pref}B1",
    )
    df_norm = agregar_deltas(df_norm, salto=2, prefix=pref, clase_col=None)

print(df_norm.select(
    ["product_id", "customer_id", "periodo", "cat1"]
    + [f"c1_tn{k}_norm" for k in range(3)]
    + [f"c1_tn{k}_delta" for k in range(2)]
    + [f"c2_tn{k}_norm" for k in range(3)]
))


shape: (96, 12)
┌────────────┬────────────┬─────────┬───────┬───┬────────────┬────────────┬────────────┬───────────┐
│ product_id ┆ customer_i ┆ periodo ┆ cat1  ┆ … ┆ c1_tn1_del ┆ c2_tn0_nor ┆ c2_tn1_nor ┆ c2_tn2_no │
│ ---        ┆ d          ┆ ---     ┆ ---   ┆   ┆ ta         ┆ m          ┆ m          ┆ rm        │
│ i64        ┆ ---        ┆ i64     ┆ str   ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│            ┆ i64        ┆         ┆       ┆   ┆ f64        ┆ f64        ┆ f64        ┆ f64       │
╞════════════╪════════════╪═════════╪═══════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ 20005      ┆ 10004      ┆ 201910  ┆ FOODS ┆ … ┆ 0.190426   ┆ 1.332808   ┆ 1.238706   ┆ 0.867791  │
│ 20008      ┆ 10005      ┆ 201910  ┆ HC    ┆ … ┆ -0.034516  ┆ 0.889998   ┆ 0.822641   ┆ 0.574791  │
│ 20004      ┆ 10009      ┆ 201910  ┆ FOODS ┆ … ┆ 0.190426   ┆ 1.332808   ┆ 1.238706   ┆ 0.867791  │
│ 20002      ┆ 10001      ┆ 201910  ┆ HC    ┆ … ┆ -0.034516  ┆ 0.984056   ┆

In [22]:
def agregar_racha(df, prefix="", alias=None, suffix="_delta"):
    """
    Agrega una columna de RACHA: hace cuantos periodos seguidos viene subiendo (+)
    o bajando (-) la serie. Se lee sobre el signo de las columnas
    {prefix}tn*{suffix} que ya dejo agregar_deltas. No elimina nada, solo agrega.

      {prefix}tn_racha = direccion * largo

    Arranca en {prefix}tn0{suffix} (el cambio mas reciente) y avanza hacia los lags
    mas viejos mientras el signo no se de vuelta:
      - el signo del primer delta != 0 fija la direccion de la racha
      - un delta de signo contrario la corta
      - un delta = 0 (empate) NO la corta: la racha sigue viva y el empate suma
      - un delta nulo (el par todavia no existia) la corta

    Casos borde:
      - todos los deltas en 0 (serie plana, tipico de un producto sin ventas)
        -> racha = 0: no hay direccion.
      - {prefix}tn0{suffix} nulo (el par no llega a salto+1 periodos de historia)
        -> racha = null, para no confundir "sin historia" con "plano".
      - como el empate no corta, una serie que sube una vez y despues queda plana
        acumula racha alta (ej. deltas [1, 0, 0, 0] -> racha 4).

    Ojo: la unidad de la racha es el salto de agregar_deltas, no el mes. Con
    salto=2, racha=+3 son 3 deltas seguidos subiendo, o sea 6 meses de ventana.

    Params
    ------
    prefix : prefijo de las columnas a leer (ej. "" para pc, "p_" para producto,
             "c1_"/"c2_"/"c3_" para categoria).
    alias  : nombre de la columna de salida. Default: "{prefix}tn_racha".
    suffix : sufijo de las columnas a leer. Default "_delta".
    """
    p, s = len(prefix), len(suffix)
    ks = sorted(
        int(c[p + 2:-s]) for c in df.columns
        if c.startswith(f"{prefix}tn") and c.endswith(suffix) and c[p + 2:-s].isdigit()
    )
    if not ks:
        raise ValueError(f"no hay columnas {prefix}tn*{suffix} en el df")
    if alias is None:
        alias = f"{prefix}tn_racha"

    sgn = {k: pl.col(f"{prefix}tn{k}{suffix}").sign() for k in ks}

    # Direccion: signo del primer delta distinto de 0. Los nulls son de cola (los
    # meses previos al first_sell del par caen en los lags mas viejos), asi que el
    # primer delta != 0 siempre cae dentro del tramo con datos.
    # Sin direccion (todo empates, o todo nulo) -> 0, y la racha termina en 0.
    d = pl.coalesce([pl.when(sgn[k] != 0).then(sgn[k]) for k in ks]).fill_null(0).cast(pl.Int32)

    # Corta en el primer lag nulo o de signo contrario. sgn*d < 0 da falso para los
    # empates (producto 0) y para d = 0, que es justo lo que queremos.
    corta = {k: sgn[k].is_null() | ((sgn[k] * d) < 0) for k in ks}

    # Largo = indice del primer corte; si nunca corta, entra toda la ventana.
    largo = pl.min_horizontal([
        pl.when(corta[k]).then(pl.lit(k, dtype=pl.Int32)).otherwise(pl.lit(len(ks), dtype=pl.Int32))
        for k in ks
    ])

    return df.with_columns(
        pl.when(pl.col(f"{prefix}tn0{suffix}").is_null())
        .then(None)
        .otherwise(d * largo)
        .cast(pl.Int32)
        .alias(alias)
    )


# --- Uso: una racha por nivel (producto-cliente, producto y las 3 categorias) ---
RACHA_PREFIXES = ["", "p_", "c1_", "c2_", "c3_"]

for pref in RACHA_PREFIXES:
    df_norm = agregar_racha(df_norm, prefix=pref)

print(df_norm.select(
    ["product_id", "customer_id", "periodo"]
    + [f"tn{k}_delta" for k in range(3)]
    + [f"{pref}tn_racha" for pref in RACHA_PREFIXES]
))

shape: (96, 11)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ product_id ┆ customer_ ┆ periodo ┆ tn0_delta ┆ … ┆ p_tn_rach ┆ c1_tn_rac ┆ c2_tn_rac ┆ c3_tn_rac │
│ ---        ┆ id        ┆ ---     ┆ ---       ┆   ┆ a         ┆ ha        ┆ ha        ┆ ha        │
│ i64        ┆ ---       ┆ i64     ┆ f64       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ i64       ┆         ┆           ┆   ┆ i32       ┆ i32       ┆ i32       ┆ i32       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 20005      ┆ 10004     ┆ 201910  ┆ 0.519593  ┆ … ┆ 2         ┆ 2         ┆ 3         ┆ 2         │
│ 20008      ┆ 10005     ┆ 201910  ┆ -0.024523 ┆ … ┆ 1         ┆ 1         ┆ 1         ┆ 1         │
│ 20004      ┆ 10009     ┆ 201910  ┆ 1.426134  ┆ … ┆ 2         ┆ 2         ┆ 3         ┆ 2         │
│ 20002      ┆ 10001     ┆ 201910  ┆ 1.424396  ┆ … ┆ 2         ┆ 1         

In [23]:
len(df_norm.columns)

393